# Fast CatBoost Balanced Submission

The full CatBoost grid is slow on this dataset. This notebook keeps the useful candidate: CatBoost with raw categorical features and balanced class weights, trained on train+validation and saved as a submission.

By default, it reuses the already-created submission file so the notebook opens and executes quickly. Set `FORCE_RETRAIN = True` to rebuild the model.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from catboost import CatBoostClassifier, Pool
from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

RANDOM_STATE = 42
ID_COL = "id"
TARGET_COL = "health_condition"
FORCE_RETRAIN = False
OUTPUT_PATH = Path("data/submission_catboost_balanced_fast.csv")


## Load Readable Feature Files

In [2]:
train_df = pd.read_csv("data/train_split_features.csv")
val_df = pd.read_csv("data/val_split_features.csv")
test_df = pd.read_csv("data/test_features.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")

full_df = pd.concat([train_df, val_df], ignore_index=True)
feature_cols = [col for col in full_df.columns if col not in [ID_COL, TARGET_COL]]
X_full = full_df[feature_cols].copy()
y_full_raw = full_df[TARGET_COL].copy()
X_test = test_df[feature_cols].copy()

cat_features = [col for col in feature_cols if X_full[col].dtype == "object"]
cat_feature_indices = [feature_cols.index(col) for col in cat_features]

label_encoder = LabelEncoder()
y_full = label_encoder.fit_transform(y_full_raw)
class_names = label_encoder.classes_.tolist()

full_pool = Pool(X_full, y_full, cat_features=cat_feature_indices, feature_names=feature_cols)
test_pool = Pool(X_test, cat_features=cat_feature_indices, feature_names=feature_cols)

print("full training:", full_df.shape)
print("test:", test_df.shape)
print("feature count:", len(feature_cols))
print("categorical features:", cat_features)
print("classes:", class_names)

full training: (690088, 58)
test: (295753, 57)
feature count: 56
categorical features: ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender', 'bmi_category']
classes: ['at-risk', 'fit', 'unhealthy']


## Train Or Reuse Balanced CatBoost Submission


In [3]:
if OUTPUT_PATH.exists() and not FORCE_RETRAIN:
    print(f"Reusing existing submission: {OUTPUT_PATH}")
    submission = pd.read_csv(OUTPUT_PATH)
else:
    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="Accuracy",
        iterations=450,
        learning_rate=0.08,
        depth=5,
        l2_leaf_reg=6.0,
        random_seed=RANDOM_STATE,
        auto_class_weights="Balanced",
        verbose=100,
        allow_writing_files=False,
    )
    model.fit(full_pool)

    test_pred = model.predict(test_pool).astype(int).ravel()
    test_labels = label_encoder.inverse_transform(test_pred)

    submission = sample_submission.copy()
    submission[ID_COL] = test_df[ID_COL].values
    submission[TARGET_COL] = test_labels
    submission.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved: {OUTPUT_PATH}")

submission_report = pd.DataFrame({
    "class": submission[TARGET_COL].value_counts().index,
    "count": submission[TARGET_COL].value_counts().values,
    "pct": submission[TARGET_COL].value_counts(normalize=True).mul(100).round(2).values,
})
submission_report


Reusing existing submission: data/submission_catboost_balanced_fast.csv


,class,count,pct
0,at-risk,223560,75.59
1,unhealthy,41459,14.02
2,fit,30734,10.39
